# NOTS-NIDS: Run All Experiments
Orchestrates all 5 experiments with checkpointing and timing.

**Sections:**
1. Load config and data
2. Preprocess and window
3. Fit NOTS detector
4. Exp 1: Baseline detection
5. Exp 2: White-box attack + theorem validation
6. Exp 3: Black-box transfer attack
7. Exp 4: Ablation study
8. Exp 5: Efficiency benchmarks
9. Generate all paper figures

In [ ]:
# ── Section 0: Setup ──────────────────────────────────────────────────
import sys, os, time, random, logging
import numpy as np
import torch

# Path setup
try:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_DIR = '/content/drive/MyDrive/nots_nids'
except ImportError:
    REPO_DIR = os.path.dirname(os.getcwd())

sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

# Reproducibility
np.random.seed(42)
random.seed(42)
torch.manual_seed(42)

# Logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger('nots')

from config import Config
cfg = Config()
print(f'Config loaded. Seed={cfg.RANDOM_SEED}')

In [ ]:
# ── Section 1: Load Data ──────────────────────────────────────────────
t_start = time.time()
from preprocessing.loader import load_cicids2017

DATA_DIR = 'data/cicids2017'  # Adjust path as needed
df, label_col = load_cicids2017(DATA_DIR)
print(f'Loaded: {df.shape}, label_col={label_col}')
print(f'Time: {time.time()-t_start:.1f}s')

In [ ]:
# ── Section 2: Preprocess and Window ─────────────────────────────────
from preprocessing.cleaner import clean_dataframe, encode_labels
from preprocessing.scaler import fit_scaler, apply_scaler
from preprocessing.windowing import create_windows
from sklearn.model_selection import train_test_split

# Clean
df_clean, dropped_cols = clean_dataframe(df, label_col)
df_clean, label_map = encode_labels(df_clean, label_col)

# Feature columns
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()
feature_cols = [c for c in numeric_cols if c not in ['label_int', 'is_attack']]
print(f'Features: {len(feature_cols)}')

# Split
idx = np.arange(len(df_clean))
idx_train, idx_temp = train_test_split(idx, test_size=1-cfg.TRAIN_RATIO, random_state=cfg.RANDOM_SEED, stratify=df_clean['is_attack'])
idx_val, idx_test = train_test_split(idx_temp, test_size=cfg.TEST_RATIO/(cfg.VAL_RATIO+cfg.TEST_RATIO), random_state=cfg.RANDOM_SEED, stratify=df_clean.iloc[idx_temp]['is_attack'])

df_train = df_clean.iloc[idx_train].reset_index(drop=True)
df_val = df_clean.iloc[idx_val].reset_index(drop=True)
df_test = df_clean.iloc[idx_test].reset_index(drop=True)

# Scale
X_train_raw = df_train[feature_cols].values
X_train_scaled, scaler = fit_scaler(X_train_raw)
df_train[feature_cols] = X_train_scaled
df_val[feature_cols] = apply_scaler(df_val[feature_cols].values, scaler)
df_test[feature_cols] = apply_scaler(df_test[feature_cols].values, scaler)

# Window
train_windows = create_windows(df_train, feature_cols, window_size=cfg.WINDOW_SIZE)
val_windows = create_windows(df_val, feature_cols, window_size=cfg.WINDOW_SIZE)
test_windows = create_windows(df_test, feature_cols, window_size=cfg.WINDOW_SIZE)
print(f'Windows: train={len(train_windows)}, val={len(val_windows)}, test={len(test_windows)}')

In [ ]:
# ── Section 3: Fit NOTS Detector ─────────────────────────────────────
from detector.nots_detector import NOTSDetector

detector = NOTSDetector(cfg)
detector.fit(train_windows, val_windows, feature_cols=feature_cols)
print(f'ε_min = {detector.epsilon_min:.6f}')
print(f'τ = {detector.tau:.6f}')
detector.save(cfg.RESULTS_DIR)
print('Detector saved.')

In [ ]:
# ── Section 4: Experiment 1 — Baseline Detection ────────────────────
from experiments.exp1_baseline import run_experiment_1
from baselines.rf_baseline import RFBaseline
from baselines.kitsune_baseline import KitsuneBaseline

# Train baselines on flow-level features
X_test = df_test[feature_cols].values
y_test = df_test['is_attack'].values
X_train_bl = df_train[feature_cols].values
y_train_bl = df_train['is_attack'].values

rf = RFBaseline(random_state=cfg.RANDOM_SEED)
rf.fit(X_train_bl, y_train_bl)

kitsune = KitsuneBaseline(n_clusters=cfg.KITSUNE_N_CLUSTERS, epochs=cfg.KITSUNE_EPOCHS, random_state=cfg.RANDOM_SEED)
X_benign_train = X_train_bl[y_train_bl == 0]
X_val_bl = df_val[feature_cols].values
y_val_bl = df_val['is_attack'].values
kitsune.fit(X_benign_train, X_val=X_val_bl, y_val=y_val_bl)

baselines = {'RF': rf, 'Kitsune': kitsune}
exp1 = run_experiment_1(detector, test_windows, baselines, cfg, X_test=X_test, y_test=y_test, label_map=label_map)

import pandas as pd
pd.DataFrame([{k: v.get('overall', v) if isinstance(v, dict) else v for k, v in exp1.items()}]).to_csv(f'{cfg.RESULTS_DIR}/exp1_results.csv')
print('Saved to Drive.')

In [ ]:
# ── Section 5: Experiment 2 — White-box Attack ──────────────────────
from experiments.exp2_whitebox import run_experiment_2

exp2 = run_experiment_2(detector, test_windows, cfg)
print('\n=== THEOREM VALIDATION ===')
print(exp2['theorem_validation']['summary'])

In [ ]:
# ── Section 6: Experiment 3 — Black-box Attack ──────────────────────
from experiments.exp3_blackbox import run_experiment_3

exp3 = run_experiment_3(detector, train_windows, test_windows, cfg)
print(f'Black-box DR: {exp3["DR"]:.4f}')

In [ ]:
# ── Section 7: Experiment 4 — Ablation Study ────────────────────────
from experiments.exp4_ablation import run_experiment_4

exp4 = run_experiment_4(train_windows, val_windows, test_windows, cfg, full_detector=detector)

In [ ]:
# ── Section 8: Experiment 5 — Efficiency ─────────────────────────────
from experiments.exp5_efficiency import run_experiment_5

exp5 = run_experiment_5(detector, cfg)

In [ ]:
# ── Section 9: Generate Paper Figures ────────────────────────────────
from evaluation.plotter import (
    plot_ablation_bars,
    plot_efficiency_table,
    plot_wasserstein_vs_delta,
)

fig_dir = cfg.FIGURES_DIR

# Ablation bars
plot_ablation_bars(exp4, f'{fig_dir}/ablation_bars')

# Efficiency table
if 'table' in exp5:
    plot_efficiency_table(exp5['table'], f'{fig_dir}/efficiency_table')

# DR vs delta (theorem validation figure)
if 'sweep' in exp2:
    nots_sweep = {d: m for d, m in exp2['sweep'].items()}
    plot_wasserstein_vs_delta({'NOTS': nots_sweep}, detector.epsilon_min, f'{fig_dir}/dr_vs_delta')

print(f'All figures saved to {fig_dir}')
print('\n✓ All experiments complete!')